# ITT Roosevelt

Notebook base para calcular el Índice de Transformación Territorial de **ITT Roosevelt**.

Flujo sugerido:

1. Configurar parámetros.
2. Cargar datos.
3. Validar columnas, fechas y coordenadas.
4. Procesar información geoespacial.
5. Calcular indicadores.
6. Normalizar scores.
7. Calcular dimensiones.
8. Calcular ITT.
9. Exportar resultados.

In [ ]:
# Parámetros principales

ZONA_ID = "itt_roosevelt"
ZONA_NOMBRE = "ITT Roosevelt"

BASE_DATA = "../data/itt_roosevelt"
BASE_OUTPUT = "../outputs/itt_roosevelt"

ANIOS = [2023, 2024, 2025]

PESOS = {
    "Seguridad": 0.30,
    "Movilidad": 0.25,
    "Entorno_Urbano": 0.20,
    "Educacion_Desarrollo": 0.13,
    "Cohesion_Social": 0.12
}

REF_ENTORNO_U = 39.2
REF_EDUC_DES = 54.9
REF_VULNERABILIDAD = 54.1

CRS_LOCAL = "EPSG:3115"
CRS_GEO = "EPSG:4326"

print(f"Configuración cargada para {ZONA_NOMBRE}")

In [ ]:
# Importaciones base

import os
from pathlib import Path

import pandas as pd
import numpy as np

try:
    import geopandas as gpd
except ImportError:
    gpd = None
    print("geopandas no está instalado. Instalar con: pip install geopandas")

BASE_DATA = Path(BASE_DATA)
BASE_OUTPUT = Path(BASE_OUTPUT)
BASE_OUTPUT.mkdir(parents=True, exist_ok=True)

print("Carpeta de datos:", BASE_DATA)
print("Carpeta de salida:", BASE_OUTPUT)

## 1. Carga de datos

En esta sección se deben cargar los GeoJSON o archivos fuente de la zona.

In [ ]:
# Ejemplo de carga de archivos disponibles en la carpeta de datos

if BASE_DATA.exists():
    archivos = list(BASE_DATA.glob("*"))
    print("Archivos encontrados:")
    for archivo in archivos:
        print("-", archivo.name)
else:
    print("No existe la carpeta de datos:", BASE_DATA)

## 2. Validación de datos

Validar columnas de fecha, coordenadas, CRS y campos necesarios para el cálculo.

In [ ]:
# Función auxiliar para normalización min-max

def clamp(value, min_value=0, max_value=100):
    return max(min_value, min(max_value, value))

def score_minmax(x, xmin, xmax, inverse=False):
    if xmax == xmin:
        return np.nan
    score = ((x - xmin) / (xmax - xmin)) * 100
    score = clamp(score)
    return 100 - score if inverse else score

## 3. Cálculo de indicadores y dimensiones

Aquí se debe adaptar la lógica del notebook principal según los datos disponibles.

In [ ]:
# Plantilla mínima de ejemplo para estructurar resultados por año

df_base = pd.DataFrame({
    "anio": ANIOS,
    "score_seguridad": [np.nan] * len(ANIOS),
    "score_movilidad": [np.nan] * len(ANIOS),
    "score_entorno_urbano": [REF_ENTORNO_U] * len(ANIOS),
    "score_educacion_desarrollo": [REF_EDUC_DES] * len(ANIOS),
    "score_cohesion_social": [np.nan] * len(ANIOS),
})

df_base

In [ ]:
# Cálculo del ITT

df_base["itt"] = (
    PESOS["Seguridad"] * df_base["score_seguridad"].fillna(0) +
    PESOS["Movilidad"] * df_base["score_movilidad"].fillna(0) +
    PESOS["Entorno_Urbano"] * df_base["score_entorno_urbano"].fillna(0) +
    PESOS["Educacion_Desarrollo"] * df_base["score_educacion_desarrollo"].fillna(0) +
    PESOS["Cohesion_Social"] * df_base["score_cohesion_social"].fillna(0)
)

df_base

## 4. Exportación de resultados

Exportar los resultados del ITT de la zona.

In [ ]:
# Exportar resultado base a Excel y CSV

excel_path = BASE_OUTPUT / f"{ZONA_ID}_resultados_itt.xlsx"
csv_path = BASE_OUTPUT / f"{ZONA_ID}_resumen_itt.csv"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_base.to_excel(writer, sheet_name="Resumen_ITT", index=False)

df_base.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Archivos exportados:")
print("-", excel_path)
print("-", csv_path)

## 5. Notas de ajuste

Este notebook es una plantilla. Para el cálculo real, adaptar las celdas de carga, spatial join, normalización y visualización con base en los archivos fuente de la zona.